# VinDr-CXR-VQA — run a trained MedGemma adapter on Kaggle

Inference only: no training happens here. Give it the paths the training run
(`notebooks/kaggle_train.ipynb`) left behind and it loads the weights **once**,
then

1. answers a few records from a split and draws the predicted box against the
   ground truth,
2. answers an image and a question of your own,
3. generates over a whole split and scores Tables 4-6 on it,
4. renders the Figure 2 overlays.

**The four paths.** Paste them into the cells below; every one can be left
empty, in which case the cell searches the attached inputs instead.

| Variable | What to paste | Where it comes from |
|---|---|---|
| `BUNDLE` | code + built JSONL | the Kaggle Dataset from `scripts/make_kaggle_bundle.py` |
| `IMAGES` | the chest X-rays | the same dataset, or the `--part images` one |
| `MODEL_PATH` | MedGemma base weights | *Add Input → Models* |
| `ADAPTER_PATH` | the LoRA adapter | the training notebook's output — `medgemma-lora/` or `medgemma-lora.zip` |

`ADAPTER_PATH` accepts a directory, one of its `checkpoint-N` subdirectories,
or the `.zip` the training notebook saves (it is unzipped for you). Set it to
`None` to run the **zero-shot baseline** — the same model without the adapter,
which is the "Pretrained" row of the paper's tables.

**Settings.** Accelerator → any GPU; on 16 GB (T4/P100) the base weights are
loaded in 4-bit, which needs `bitsandbytes`. Internet on or off is detected
automatically, and with it off everything — code, images, base weights,
adapter — must be attached.

> **Import this notebook before attaching your inputs.** Kaggle applies a
> notebook's stored settings on import; this one stores none, but an import on
> top of a configured notebook can still clear the panel.

**Cost.** Loading the weights takes a few minutes; after that a single answer
is a couple of seconds, and a full 2,637-pair validation pass is 1.5-2 h on a
4-bit T4. `N_EVAL` in section 5 caps it.

## 0. Setup

In [ ]:
import os, sys, glob, json, shutil, subprocess

# ---------------------------------------------------------------------------
# Paths from the training run. Leave empty to search the attached inputs.
#   BUNDLE  the code (and the built JSONL, unless it is in a separate input)
#   IMAGES  the chest X-rays, when they were attached as their own dataset
# e.g. "/kaggle/input/datasets/<user>/vqa-repo"
BUNDLE = ""
IMAGES = ""
# ---------------------------------------------------------------------------

def find_inputs(marker, root="/kaggle/input", max_depth=6):
    """Every attached directory containing `marker`, shallowest first.

    Kaggle nests inputs differently depending on how they were attached
    (/kaggle/input/<slug> for some, /kaggle/input/datasets/<user>/<slug> or
    /kaggle/input/models/<user>/<model>/<fw>/<var>/<ver>/ for others), so this
    walks rather than globbing one level.
    """
    hits, base = [], root.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.count("/") - base >= max_depth:
            dirnames[:] = []
            continue
        # Never descend into the image tree; there are thousands of files.
        dirnames[:] = [d for d in dirnames if d != "train"]
        if os.path.exists(os.path.join(dirpath, marker)):
            hits.append(dirpath)
            dirnames[:] = []
    return sorted(hits, key=len)

def tree(root="/kaggle/input", max_depth=4, limit=40):
    """What is actually attached, files included.

    Two levels of directory names say nothing about why a search failed --
    an un-extracted zip and a dataset that was never attached look identical.
    Showing the files tells the two apart at a glance.
    """
    lines, seen, base = [], 0, root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count("/") - base
        if depth >= max_depth:
            dirnames[:] = []
        # Never list the image tree; there are thousands of files.
        dirnames[:] = [d for d in dirnames if d != "train"]
        lines.append("  " * depth + (os.path.basename(dirpath) or root) + "/")
        for name in sorted(filenames)[:5]:
            lines.append("  " * (depth + 1) + name)
        if len(filenames) > 5:
            lines.append("  " * (depth + 1) + f"… {len(filenames) - 5} file khác")
        seen += 1
        if seen >= limit:
            lines.append("  …")
            break
    return "\n".join(lines)

def unpack(marker, dest="/kaggle/working/unpacked"):
    """Extract an attached .zip that carries `marker`, if one is still zipped.

    Uploading a bundle through the Kaggle UI usually extracts it, but not
    always -- and a dataset left as a single .zip is indistinguishable from
    one that was never attached. Rather than stop on a zip holding exactly
    what is being looked for, unpack it into the writable tree and go on.
    """
    import zipfile
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        dirnames[:] = [d for d in dirnames if d != "train"]
        for name in filenames:
            if not name.endswith(".zip"):
                continue
            path = os.path.join(dirpath, name)
            try:
                with zipfile.ZipFile(path) as z:
                    inside = z.namelist()
                    if not any(n.startswith(marker) for n in inside):
                        continue
                    out = os.path.join(dest, os.path.splitext(name)[0])
                    # Code and JSONL ride in the same bundle, so this is
                    # reached twice; unpack it once.
                    if os.path.exists(os.path.join(out, marker)):
                        return out
                    print(f"{path} chưa được giải nén; đang bung "
                          f"{len(inside)} file -> {out}")
                    z.extractall(out)
                    return out
            except zipfile.BadZipFile:
                continue
    return None

def locate(marker, what):
    """Every input holding `marker`, unpacking a stray zip if that is all there is."""
    hits = find_inputs(marker)
    if hits:
        return hits
    out = unpack(marker)
    if out:
        return find_inputs(marker, root=out)
    raise SystemExit(
        f"Không input nào chứa {marker} ({what}).\n\n"
        f"Đang attach:\n{tree()}\n\n"
        "Kiểm tra: dataset đã attach chưa, đã trỏ sang version mới nhất chưa, "
        "và Kaggle đã giải nén file .zip chưa.")

# Checking that the .py files merely exist is not enough: an attached dataset
# can carry an older copy of a module that is present but missing a fix, and
# the failure then surfaces half an hour later as a traceback from deep inside
# a library. So look for a snippet each fix introduced.
#
# Snippets, not just function names: some fixes add no new function -- the one
# that moves the box loss onto the GPU is a single line inside an existing
# method -- and a names-only check waves those through.
NEEDS = {
    "modeling":  ["def load_base_model", "def disable_broken_peft_dispatchers",
                  "def adapt_training_kwargs", "def language_side_targets"],
    "evaluate":  ["def diagnose_predictions", "def load_predictions",
                  "def eval_boxes"],
    "train":     ["def latest_checkpoint", "def warn_if_nothing_will_be_saved",
                  "def build_training_arguments"],
    "losses":    ["self.to(logits.device)"],
    "results":   ["def render"],
    "preflight": ["def check_model"],
    "infer":     ["def run"],
    "boxes":     ["def to_canvas", "def match_any", "def greedy_match"],
}

def missing_markers(root):
    """Which of NEEDS a copy of the package under `root` does not have."""
    gone = []
    for mod, markers in NEEDS.items():
        path = os.path.join(root, "vindr_cxr_vqa", mod + ".py")
        if not os.path.exists(path):
            gone.append(f"{mod}.py is missing entirely")
            continue
        src = open(path, encoding="utf-8").read()
        gone += [f"{mod}.py: {m}" for m in markers if m not in src]
    return gone

NEEDS["infer"] = ["def run", "def answer", "def load_processor"]

# The code, the built JSONL and the images may live in one input or in two.
#
# Two attached inputs can both carry vindr_cxr_vqa/ -- an older full bundle
# next to a freshly uploaded code-only dataset -- and picking by path length
# alone is a coin flip between them. So prefer whichever copy actually has the
# functions this notebook needs. When none does, keep the shallowest, so the
# report below names what is missing rather than running yesterday's code.
if BUNDLE:
    CODE_ROOT = BUNDLE
else:
    roots = locate("vindr_cxr_vqa", "code")
    CODE_ROOT = next((r for r in roots if not missing_markers(r)), roots[0])
    if len(roots) > 1:
        print(f"{len(roots)} attached inputs carry vindr_cxr_vqa/; "
              f"chose {CODE_ROOT}")

PROCESSED_ROOT = locate("data/processed/train.jsonl", "JSONL đã build")[0]
IMAGES_ROOT = IMAGES or locate("dataset/train_data/train", "ảnh X-quang")[0]

print("code     :", CODE_ROOT)
print("jsonl    :", PROCESSED_ROOT)
print("images   :", IMAGES_ROOT)

# /kaggle/input is read-only, so work from a writable copy of the code.
WORK = "/kaggle/working/vindr"
os.makedirs(WORK, exist_ok=True)
for item in ("vindr_cxr_vqa", "configs", "requirements.txt", "docs"):
    src = os.path.join(CODE_ROOT, item)
    if not os.path.exists(src):
        continue
    dst = os.path.join(WORK, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

os.chdir(WORK)
sys.path.insert(0, WORK)
print("\ncwd:", os.getcwd())
print(sorted(os.listdir(WORK)))

# Fail now, clearly, if the attached code predates what this notebook needs.
# This re-checks the writable copy rather than CODE_ROOT, so a half-finished
# copy is caught too -- and it is the only report when BUNDLE was set by hand
# or when no attached input is current.
stale = missing_markers(WORK)
if stale:
    raise SystemExit(
        "The attached code is OUT OF DATE. Missing:\n  "
        + "\n  ".join(stale)
        + f"\n\nIt was read from: {CODE_ROOT}\n"
        "Rebuild with `python scripts/make_kaggle_bundle.py --part code`, "
        "upload it as a new version of that dataset, and re-run.")
print("code check: up to date")

In [ ]:
# None = detect automatically; set True/False to force it.
# Detection matters: with Internet off but OFFLINE false, pip spends minutes
# retrying DNS before giving up.
OFFLINE = None

def has_internet(host="pypi.org", port=443, timeout=4):
    import socket
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

if OFFLINE is None:
    OFFLINE = not has_internet()
    print(f"detected: {'no internet' if OFFLINE else 'internet available'}")

if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("offline mode: no pip install, no downloads, no hub lookups")
else:
    print("online mode: packages and weights may be fetched")

In [ ]:
# MedGemma is a Gemma 3 model: it needs a recent transformers.
# With Internet off this is skipped — the preflight cell below checks whether
# Kaggle's preinstalled versions are new enough.
# --retries/--timeout keep a misdetected network from stalling for minutes.
if not OFFLINE:
    %pip install -q -U --retries 1 --timeout 15 \
        "transformers>=4.50" "accelerate>=1.4" "peft>=0.14" \
        "bitsandbytes>=0.45" pyyaml

import torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"vram         {total/2**30:.1f} GB total, {free/2**30:.1f} GB free")
    print("bf16         ", torch.cuda.is_bf16_supported())

In [ ]:
# HF_TOKEN is only needed to DOWNLOAD the gated weights. With the model
# attached from Kaggle, or offline, it is not used at all.
if not OFFLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
    except Exception as exc:
        print("No HF_TOKEN secret:", exc)
        print("Fine if you attach the model instead of downloading it.")

os.environ.setdefault("HF_HOME", "/kaggle/working/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

### Where the model weights come from

**Attach MedGemma via *Add Input → Models*, then paste its path into
`MODEL_PATH` below.** Kaggle shows the path on the model's panel in the right
sidebar once it is attached — something like
`/kaggle/input/medgemma/transformers/4b-it/1`.

Leaving `MODEL_PATH` empty is fine too: the cell then searches every attached
input for a directory holding `config.json` plus `*.safetensors`, and falls
back to downloading from HuggingFace when the notebook is online.

With Internet **off** the weights must already be attached — a Kaggle notebook
cannot reach the hub. If no MedGemma model is available to attach, run
`python scripts/fetch_model.py` on a machine with internet (≈8 GB, gated) and
upload the result as a Dataset.

In [ ]:
# ---------------------------------------------------------------------------
# Paste the attached model path here. Add Input -> Models shows it in the
# right-hand panel, e.g.
#   "/kaggle/input/models/<user>/googlemedgemma-1-5-4b-it/transformers/default/1/medgemma-1.5-4b-it"
# Leave empty to search the attached inputs automatically.
MODEL_PATH = ""
# ---------------------------------------------------------------------------

def find_model_dir(root="/kaggle/input"):
    """Any attached directory holding config.json plus safetensors weights."""
    hits = []
    for dirpath, dirnames, files in os.walk(root):
        # The data bundle holds thousands of PNGs; nothing to find in there.
        dirnames[:] = [d for d in dirnames if d != "train"]
        if "config.json" in files and any(f.endswith(".safetensors")
                                          for f in files):
            hits.append(dirpath)
            dirnames[:] = []
    # Prefer a path naming medgemma, then the one with the most weight shards.
    hits.sort(key=lambda p: ("medgemma" not in p.lower(),
                             -len([f for f in os.listdir(p)
                                   if f.endswith(".safetensors")])))
    return hits[0] if hits else None

if MODEL_PATH:
    if not os.path.isdir(MODEL_PATH):
        raise SystemExit(f"MODEL_PATH does not exist: {MODEL_PATH}\n"
                         f"attached inputs: {os.listdir('/kaggle/input')}")
    print("using MODEL_PATH:", MODEL_PATH)
else:
    MODEL_PATH = find_model_dir()
    if MODEL_PATH:
        print("auto-detected model:", MODEL_PATH)
    elif OFFLINE:
        raise SystemExit(
            "Offline mode needs the weights attached, but nothing under "
            "/kaggle/input holds config.json + *.safetensors.\n"
            "Attach MedGemma via Add Input -> Models and set MODEL_PATH.")
    else:
        MODEL_PATH = "google/medgemma-4b-it"
        print("no attached weights; will download", MODEL_PATH)

if os.path.isdir(MODEL_PATH):
    print(sorted(f for f in os.listdir(MODEL_PATH)
                 if not f.startswith("."))[:12])

### The adapter

What training produced. `/kaggle/working` is wiped when a session ends, so the
adapter is normally reached one of two ways:

* **the training notebook's output** — *Add Input → Notebook Output*, which
  gives a path like `/kaggle/input/<notebook-slug>/medgemma-lora.zip` or a
  `medgemma-lora/` directory;
* **a Dataset** you uploaded the downloaded `medgemma-lora.zip` to.

Either works. A `checkpoint-N` directory works too — that is how you compare
step 300 against step 600 without retraining. `ADAPTER_PATH = None` skips the
adapter entirely and gives the zero-shot baseline.

In [ ]:
# ---------------------------------------------------------------------------
# The LoRA adapter to load. Accepts a directory, a checkpoint-N inside one, or
# the .zip the training notebook writes. "" searches the attached inputs;
# None runs the zero-shot baseline (no adapter at all).
ADAPTER_PATH = ""
# ---------------------------------------------------------------------------

WEIGHT_FILES = ("adapter_model.safetensors", "adapter_model.bin")

def is_adapter(path):
    """A directory PEFT can load: a config plus weights next to it."""
    return (os.path.isdir(path)
            and os.path.exists(os.path.join(path, "adapter_config.json"))
            and any(os.path.exists(os.path.join(path, f))
                    for f in WEIGHT_FILES))

def step_of(path):
    """The N in checkpoint-N, or -1 for anything else."""
    name = os.path.basename(path.rstrip("/"))
    tail = name.rsplit("-", 1)[-1]
    return int(tail) if name.startswith("checkpoint-") and tail.isdigit() else -1

def newest_checkpoint(path):
    """The highest-numbered checkpoint-N under `path`, when the run dir itself
    holds no weights -- an interrupted session saves checkpoints but never
    writes the final adapter."""
    cks = [p for p in glob.glob(os.path.join(path, "checkpoint-*"))
           if is_adapter(p)]
    return max(cks, key=step_of) if cks else None

def unzip_adapter(path, dest="/kaggle/working/adapter"):
    import zipfile
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(dest)
    print(f"unzipped {path} -> {dest}")
    # make_archive zips the *contents*, but a hand-made zip may hold a folder.
    return dest if is_adapter(dest) or newest_checkpoint(dest) else next(
        (os.path.join(dest, d) for d in sorted(os.listdir(dest))
         if is_adapter(os.path.join(dest, d))), dest)

def find_adapter(root="/kaggle/input"):
    """An attached adapter directory, or a zip holding one."""
    hits = []
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "train"]
        if "adapter_config.json" in files and is_adapter(dirpath):
            hits.append(dirpath)
    if hits:
        # Prefer a final adapter over a checkpoint; among checkpoints, the
        # furthest-trained one. Shallowest path breaks any remaining tie.
        return sorted(hits, key=lambda p: (step_of(p) >= 0, -step_of(p),
                                           len(p)))[0]
    import zipfile
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "train"]
        for name in files:
            if not name.endswith(".zip"):
                continue
            path = os.path.join(dirpath, name)
            try:
                with zipfile.ZipFile(path) as z:
                    if any(n.endswith("adapter_config.json")
                           for n in z.namelist()):
                        return unzip_adapter(path)
            except zipfile.BadZipFile:
                continue
    return None

if ADAPTER_PATH is None:
    print("no adapter: zero-shot baseline (the paper's 'Pretrained' row)")
else:
    if ADAPTER_PATH.endswith(".zip"):
        ADAPTER_PATH = unzip_adapter(ADAPTER_PATH)
    elif not ADAPTER_PATH:
        ADAPTER_PATH = find_adapter()
        if ADAPTER_PATH:
            print("auto-detected adapter:", ADAPTER_PATH)
    if ADAPTER_PATH and not is_adapter(ADAPTER_PATH):
        nested = newest_checkpoint(ADAPTER_PATH)
        if not nested:
            raise SystemExit(
                f"{ADAPTER_PATH} is not a LoRA adapter: no adapter_config.json "
                f"+ {' / '.join(WEIGHT_FILES)} in it, and no checkpoint-N "
                f"under it.\n\nĐang attach:\n{tree()}\n\n"
                "Attach the training notebook's output (Add Input -> Notebook "
                "Output), or set ADAPTER_PATH = None for the zero-shot run.")
        print(f"{ADAPTER_PATH} holds no final adapter; using {nested}")
        ADAPTER_PATH = nested
    if not ADAPTER_PATH:
        raise SystemExit(
            "No adapter found under /kaggle/input.\n\n"
            f"Đang attach:\n{tree()}\n\n"
            "Attach the training notebook's output, paste its path into "
            "ADAPTER_PATH, or set ADAPTER_PATH = None to run zero-shot.")

if ADAPTER_PATH:
    print("adapter  :", ADAPTER_PATH)
    acfg = json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json")))
    print(json.dumps({k: acfg.get(k) for k in
                      ("base_model_name_or_path", "r", "lora_alpha",
                       "lora_dropout", "target_modules")}, indent=2,
                     default=list))
    size = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f))
               for f in os.listdir(ADAPTER_PATH)
               if os.path.isfile(os.path.join(ADAPTER_PATH, f)))
    print(f"size     : {size / 2**20:.1f} MB")

## 1. Config

In [ ]:
# One config for this session: the attached data, the attached weights, and a
# batch size the GPU can hold. Nothing here trains, so only the loading knobs
# of the train section are used.
import torch, yaml

CFG_PATH = os.path.join(WORK, "configs", "kaggle_infer.yaml")
with open(os.path.join(WORK, "configs", "default.yaml")) as fh:
    cfg = yaml.safe_load(fh)

cfg["paths"]["dataset_dir"]   = os.path.join(IMAGES_ROOT, "dataset")
cfg["paths"]["processed_dir"] = os.path.join(PROCESSED_ROOT, "data", "processed")
cfg["paths"]["output_dir"]    = "/kaggle/working/outputs"
cfg["model"]["name"] = MODEL_PATH

# Generation needs far less memory than training, so quantize only to fit:
# 4-bit below 24 GB, full precision above it. Keep this the same as the run
# that produced the adapter if you are comparing numbers -- 4-bit weights move
# the answers a little.
VRAM = (torch.cuda.get_device_properties(0).total_memory / 2**30
        if torch.cuda.is_available() else 0)
FOUR_BIT = VRAM < 24
INFER_BATCH = 16 if VRAM >= 40 else (8 if VRAM >= 24 else 2)
cfg["train"]["load_in_4bit"] = FOUR_BIT
print(f"{VRAM:.0f} GB: 4-bit={FOUR_BIT}, batch {INFER_BATCH}, "
      f"max_new_tokens {cfg['eval']['max_new_tokens']}")

with open(CFG_PATH, "w") as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)

print(json.dumps({"paths": cfg["paths"], "model": cfg["model"],
                  "data": cfg["data"], "generate": cfg["generate"]}, indent=2))
assert os.path.isdir(cfg["paths"]["dataset_dir"]), cfg["paths"]["dataset_dir"]
assert os.path.isdir(cfg["paths"]["processed_dir"]), cfg["paths"]["processed_dir"]

### Preflight

Checks package versions, the GPU, the attached weights and the dataset files
in one pass, so a missing piece surfaces now rather than after the four
minutes it takes to load the weights. It exits non-zero if anything is
blocking.

In [ ]:
# Stop here on a blocking problem rather than failing hours later.
r = subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.preflight",
                    "--config", CFG_PATH], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("Preflight found blocking problems — fix them above "
                     "before running the rest of the notebook.")

## 2. Load the model once

Reading 8 GB of weights takes a few minutes, so it happens here and every cell
below reuses `model` and `processor`. If you change `ADAPTER_PATH`, re-run
this cell — and only this one.

In [ ]:
import time
from vindr_cxr_vqa.config import load_config
from vindr_cxr_vqa import infer as vqa_infer

CFG = load_config(CFG_PATH)

t0 = time.time()
processor = vqa_infer.load_processor(CFG, ADAPTER_PATH)
model = vqa_infer.load_model(CFG, ADAPTER_PATH, FOUR_BIT)
print(f"\nloaded in {time.time() - t0:.0f}s on "
      f"{next(model.parameters()).device}, "
      f"{'with' if ADAPTER_PATH else 'without'} adapter")

## 3. One record, end to end

The smallest real test: take records straight out of a split, ask the model
its own question about them, and draw what came back.

* **green** — the fused ground-truth boxes for that image, every finding;
* **red** — each `<loc_...>` span the model emitted, labelled with its best
  IoU against the green boxes.

A prediction with no `<loc_...>` in it draws nothing — common for the zero-shot
baseline, which has not learned the span format.

In [ ]:
SPLIT  = "test"     # train / val / test
N_DEMO = 3          # how many records to answer here
PICK   = "first"    # "first" or "random"

import random
from PIL import Image, ImageDraw
from IPython.display import display
from vindr_cxr_vqa import boxes as B
from vindr_cxr_vqa.infer import answer, load_split, open_image

records = load_split(CFG, SPLIT)
with open(os.path.join(CFG.paths.processed_dir, "images.json"),
          encoding="utf-8") as fh:
    INDEX = json.load(fh)

# Which space the model was told to answer in; the boxes mean nothing without
# it, since <loc_300_...> is a different place in pixels than in 0-999.
SPACE = "norm1000" if CFG.data.loc_format == "norm1000" else "pixel"
UPSCALE = 3        # the stored PNGs are 224x224; enlarge so boxes stay legible
GREEN, RED = (60, 220, 60), (240, 60, 60)

def draw(image, prediction, image_id=None):
    """The X-ray with ground truth in green and every predicted box in red."""
    pic = image.resize((image.width * UPSCALE, image.height * UPSCALE),
                       Image.LANCZOS)
    d, (w, h) = ImageDraw.Draw(pic), pic.size
    meta = INDEX.get(image_id or "", {})
    # Ground truth is in original DICOM pixels, so the canvas mapping needs the
    # original size -- not the 224x224 the PNG was resized to.
    iw = meta.get("width", image.width)
    ih = meta.get("height", image.height)

    gts = []
    for box in meta.get("boxes_fused", []):
        canvas = B.to_canvas(box["bbox_orig"], "pixel", iw, ih, w, h)
        gts.append(canvas)
        d.rectangle(canvas, outline=GREEN, width=2)
        d.text((canvas[0] + 3, max(0, canvas[1] - 11)), box["class_name"],
               fill=GREEN)

    ious = []
    for box in B.extract_locs(prediction):
        canvas = B.to_canvas(box, SPACE, iw, ih, w, h)
        best = max((B.iou(canvas, g) for g in gts), default=0.0)
        ious.append(best)
        d.rectangle(canvas, outline=RED, width=2)
        d.text((canvas[0] + 3, max(0, canvas[1] - 11)), f"IoU {best:.2f}",
               fill=RED)
    display(pic)
    return ious

picked = (random.sample(records, N_DEMO) if PICK == "random"
          else records[:N_DEMO])
images = [open_image(CFG, r) for r in picked]

t0 = time.time()
predictions = answer(model, processor, CFG, picked, images)
print(f"{len(picked)} answers in {time.time() - t0:.1f}s "
      f"({(time.time() - t0) / len(picked):.1f}s each)\n")

for r, pred, img in zip(picked, predictions, images):
    print("image   :", r["image_id"])
    print("Q       :", r["question"])
    print("GT      :", r["gt_finding"], "|", r["gt_location"])
    print("answer  :", pred)
    ious = draw(img, pred, r["image_id"])
    print("boxes   :", f"{len(ious)} predicted, best IoU "
          + (f"{max(ious):.2f}" if ious else "n/a — no <loc_...> emitted"))
    print("-" * 78)

## 4. Your own image and question

The same call with nothing from the dataset behind it: any image file, any
question. This is what using the model outside this repo looks like — no
ground truth, so only the prediction is drawn.

Point `IMAGE` at any `.png`/`.jpg` on the machine (an attached input, or a file
you uploaded). Leaving it empty uses the first record of the split above.

In [ ]:
IMAGE    = ""      # e.g. f"{CFG.paths.dataset_dir}/train_data/train/<id>.png"
QUESTION = "What abnormality is present in this chest X-ray, and where is it?"

if not IMAGE:
    IMAGE = os.path.join(CFG.paths.dataset_dir, records[0]["image_path"])
    print("IMAGE is empty; using", IMAGE)
if not os.path.exists(IMAGE):
    raise SystemExit(f"no such image: {IMAGE}")

picture = Image.open(IMAGE).convert("RGB")
t0 = time.time()
# A record is just a question here: build_messages reads nothing else, so an
# image off the street goes through exactly the path the dataset does.
reply = answer(model, processor, CFG, [{"question": QUESTION}], [picture])[0]
print(f"{time.time() - t0:.1f}s\n")
print(reply, "\n")

ious = draw(picture, reply)       # no image_id => no ground truth to compare
print(f"{len(ious)} box(es) predicted")

## 5. Score a whole split

Generation over the split, then the same scorer the paper's tables come from:
Table 4 (VQA), Table 5 (box detection at IoU ≥ 0.3), Table 6 (per-image
localization quality).

`N_EVAL` caps the pairs. Capping makes the run quick but the numbers partial —
and when `SPLIT = "val"` the predictions are also scored against **test**,
which is a subset of validation, so a small cap can leave very few test pairs
in it. Set `N_EVAL = None` for the numbers to be comparable with the paper.

In [ ]:
N_EVAL = 200       # None = the whole split

OUTDIR = CFG.paths.output_dir
os.makedirs(OUTDIR, exist_ok=True)
TAG = "ft" if ADAPTER_PATH else "base"
PRED = f"{OUTDIR}/preds_{SPLIT}_{TAG}.jsonl"

t0 = time.time()
# The model is already in memory from section 2; run() takes it as it is
# rather than reading the weights a second time.
vqa_infer.run(CFG, SPLIT, ADAPTER_PATH, PRED, N_EVAL, INFER_BATCH,
              model=model, processor=processor)
print(f"generation took {(time.time() - t0) / 60:.1f} min")

In [ ]:
# A val prediction file also covers test, which is a subset of it.
SPLITS = ("val", "test") if SPLIT == "val" else (SPLIT,)

for split in SPLITS:
    r = subprocess.run(
        [sys.executable, "-m", "vindr_cxr_vqa.evaluate", "--config", CFG_PATH,
         "--pred", PRED, "--split", split,
         "--out", f"{OUTDIR}/eval_{split}_{TAG}.json"],
        capture_output=True, text=True)
    print(r.stdout or r.stderr)

In [ ]:
# Run this notebook twice -- once with ADAPTER_PATH set, once with it None --
# and both eval files are on disk, so the paper's side-by-side table can be
# rendered. With only one of the two, this cell says so and does nothing.
from IPython.display import Markdown, display

for split in SPLITS:
    base, ft = (f"{OUTDIR}/eval_{split}_base.json",
                f"{OUTDIR}/eval_{split}_ft.json")
    if not (os.path.exists(base) and os.path.exists(ft)):
        print(f"{split}: need both eval_{split}_base.json and "
              f"eval_{split}_ft.json; have "
              f"{[os.path.basename(p) for p in (base, ft) if os.path.exists(p)]}")
        continue
    subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.results",
                    "--base", base, "--ft", ft, "--split", split,
                    "--out", f"{OUTDIR}/results_{split}.md"],
                   capture_output=True, text=True)
    display(Markdown(open(f"{OUTDIR}/results_{split}.md").read()))

## 6. Figure 2 overlays

The same drawing as section 3, done in bulk by `vindr_cxr_vqa.visualize` and
written to PNGs: ground truth green, zero-shot yellow, fine-tuned red. It
picks the images where a box was actually predicted.

In [ ]:
flag = "--pred-ft" if ADAPTER_PATH else "--pred-base"
!{sys.executable} -m vindr_cxr_vqa.visualize --config {CFG_PATH} \
    {flag} {PRED} --split {SPLIT} --n 6 --out {OUTDIR}/figures

from IPython.display import Image as IPyImage
for p in sorted(glob.glob(f"{OUTDIR}/figures/*.png"))[:6]:
    print(p)
    display(IPyImage(filename=p))

## 7. What is on disk

In [ ]:
# /kaggle/working is wiped when the session ends; Save Version keeps whatever
# is listed here as this notebook's output.
for f in sorted(glob.glob(f"{OUTDIR}/*.json") + glob.glob(f"{OUTDIR}/*.md")
                + glob.glob(f"{OUTDIR}/*.jsonl")):
    print(f"  {os.path.basename(f):<28} {os.path.getsize(f) / 2**10:8.1f} KB")

print(f"\nmodel   : {MODEL_PATH}")
print(f"adapter : {ADAPTER_PATH or 'none (zero-shot)'}")
print(f"split   : {SPLIT}, {'all' if N_EVAL is None else N_EVAL} QA pairs")